# Demand Forecasting with Bedrock

## Augmenting Statistical Forecasts with Unstructured Market Signals

Traditional demand forecasting in Retail and CPG relies on time-series statistical models (ARIMA, exponential smoothing, Prophet) trained on historical sales data. These models perform well under steady-state conditions but fail to anticipate demand shifts driven by unstructured signals: social media trends, weather events, competitor actions, and supply disruptions.

This notebook demonstrates a hybrid approach:
1. Generate a baseline statistical forecast using standard methods
2. Collect unstructured market signals (simulated here with synthetic data)
3. Use a foundation model (Amazon Bedrock - Claude) to interpret those signals and produce a demand adjustment recommendation
4. Combine the statistical forecast with the model-recommended adjustment

**Prerequisites:** `boto3`, `pandas`, `numpy`, `matplotlib`

**Note:** This notebook uses synthetic data. No real customer or sales data is included.

## 1. Generate Synthetic Demand Data

We simulate 52 weeks of sales history for a CPG product (oat milk, 32oz) at a regional distribution center. The data includes:
- Base demand with weekly seasonality
- A promotional lift in weeks 12-14
- Random noise to simulate real-world variability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

np.random.seed(42)

# Generate 52 weeks of synthetic demand
weeks = 52
base_demand = 2400  # units per week
seasonality = np.sin(np.linspace(0, 2 * np.pi, weeks)) * 300  # seasonal wave
noise = np.random.normal(0, 150, weeks)  # random variability
promo_lift = np.zeros(weeks)
promo_lift[11:14] = 600  # promotional event weeks 12-14

demand = base_demand + seasonality + noise + promo_lift
demand = np.maximum(demand, 500)  # floor at 500 units

# Build dataframe
start_date = datetime(2025, 7, 1)
dates = [start_date + timedelta(weeks=i) for i in range(weeks)]

df = pd.DataFrame({
    'week_start': dates,
    'units_sold': demand.astype(int),
    'sku': 'SKU-4471-OATMILK-32OZ',
    'location': 'DC-CHI-01'
})

print(f"Dataset: {len(df)} weeks of demand history")
print(f"Mean weekly demand: {df['units_sold'].mean():.0f} units")
print(f"Std deviation: {df['units_sold'].std():.0f} units")
df.head(10)

In [ ]:
# Visualize the demand history
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['week_start'], df['units_sold'], marker='.', linewidth=1.5,
        color='#232F3E', markersize=4)
ax.axvspan(dates[11], dates[13], alpha=0.15, color='#FF9900', label='Promo period')
ax.set_xlabel('Week')
ax.set_ylabel('Units Sold')
ax.set_title('Weekly Demand History — SKU-4471-OATMILK-32OZ @ DC-CHI-01')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Baseline Statistical Forecast

We use a simple exponential smoothing model to produce a 4-week forward forecast. In production, this would be Prophet, DeepAR, or a custom time-series model. The point here is not the statistical method — it is the augmentation layer that follows.

In [ ]:
# Simple exponential smoothing for baseline forecast
def exponential_smoothing(series, alpha=0.3):
    """Single exponential smoothing."""
    result = [series.iloc[0]]
    for i in range(1, len(series)):
        result.append(alpha * series.iloc[i] + (1 - alpha) * result[-1])
    return result

# Fit on historical data
smoothed = exponential_smoothing(df['units_sold'], alpha=0.3)
last_smoothed = smoothed[-1]

# 4-week forward forecast (flat projection from last smoothed value)
forecast_weeks = 4
forecast_dates = [dates[-1] + timedelta(weeks=i+1) for i in range(forecast_weeks)]
statistical_forecast = [last_smoothed] * forecast_weeks

print(f"Statistical forecast (next 4 weeks): {last_smoothed:.0f} units/week")
print(f"This is a naive flat projection — good for stable demand, blind to market shifts.")

## 3. Collect Unstructured Market Signals

In a production system, these signals come from:
- Social media monitoring APIs (Brandwatch, Sprout Social)
- Weather forecast services
- News aggregation feeds
- Competitive intelligence platforms

Here we simulate them as structured text that a foundation model can reason over.

In [ ]:
# Simulated market signals for the forecast period
market_signals = {
    "social_media": {
        "signal": "Oat milk mentions on TikTok up 340% week-over-week in Chicago metro. "
                  "Viral recipe video from @healthychef (2.1M followers) posted June 20 "
                  "featuring oat milk cold brew. Video has 4.8M views and climbing.",
        "source": "TikTok Trends API",
        "timestamp": "2026-06-21T08:00:00Z"
    },
    "weather": {
        "signal": "Heat wave forecast for Chicago metro: 95-102°F expected June 23-30. "
                  "Historical correlation shows 8-12% increase in cold beverage demand "
                  "during sustained heat events in this region.",
        "source": "NOAA Extended Forecast",
        "timestamp": "2026-06-21T06:00:00Z"
    },
    "competitor": {
        "signal": "Primary competitor (Brand X Oat Milk 32oz) out of stock at 3 of 5 "
                  "major Chicago-area Kroger distribution points per retailer portal. "
                  "Estimated restock date: July 5.",
        "source": "Retailer Portal Scrape",
        "timestamp": "2026-06-21T10:00:00Z"
    },
    "promotional_calendar": {
        "signal": "No promotional activity scheduled for SKU-4471 in the next 4 weeks. "
                  "Retailer confirmed standard shelf pricing through July 20.",
        "source": "Trade Marketing Calendar",
        "timestamp": "2026-06-20T14:00:00Z"
    }
}

# Display signals
for signal_type, data in market_signals.items():
    print(f"\n{'='*60}")
    print(f"Signal: {signal_type.upper()}")
    print(f"Source: {data['source']}")
    print(f"Content: {data['signal']}")

## 4. Foundation Model Demand Adjustment

We send the statistical forecast along with the unstructured signals to Amazon Bedrock (Claude). The model's job is NOT to produce a forecast — it is to reason about whether the signals warrant an adjustment to the existing statistical forecast, and by how much.

This is the key architectural insight: **the foundation model is a reasoning layer, not a prediction layer.** It interprets context that statistical models cannot process.

In [ ]:
import boto3
import json

def build_adjustment_prompt(sku, location, statistical_forecast_value, signals):
    """Build the prompt for the demand adjustment model."""
    
    signal_text = ""
    for signal_type, data in signals.items():
        signal_text += f"\n### {signal_type.replace('_', ' ').title()}\n"
        signal_text += f"Source: {data['source']}\n"
        signal_text += f"Signal: {data['signal']}\n"
    
    prompt = f"""You are a demand planning analyst for a CPG company. Your job is to 
evaluate whether unstructured market signals warrant an adjustment to an existing 
statistical demand forecast.

## Current Forecast
- SKU: {sku}
- Location: {location}
- Statistical forecast (next 4 weeks): {statistical_forecast_value:.0f} units/week
- Forecast method: Exponential smoothing on 52 weeks of history

## Market Signals (collected in the last 24 hours)
{signal_text}

## Your Task
Based on these signals, recommend a demand adjustment percentage for the next 4 weeks.

Respond in this exact JSON format:
{{
  "adjustment_pct": <number between -30 and +30>,
  "confidence": <number between 0.0 and 1.0>,
  "reasoning": "<2-3 sentences explaining your recommendation>",
  "primary_driver": "<which signal is the strongest factor>",
  "risk_factors": "<what could make this adjustment wrong>"
}}

Be conservative. Only recommend adjustments when signals are strong and corroborated 
by multiple sources. A +5% adjustment on a large-volume SKU is material."""
    
    return prompt

# Build the prompt
prompt = build_adjustment_prompt(
    sku='SKU-4471-OATMILK-32OZ',
    location='DC-CHI-01',
    statistical_forecast_value=last_smoothed,
    signals=market_signals
)

print("Prompt built. Length:", len(prompt), "characters")
print("\n--- First 500 chars ---")
print(prompt[:500])

In [ ]:
# Call Amazon Bedrock
# NOTE: Requires AWS credentials with bedrock:InvokeModel permission
# If running locally without credentials, skip this cell and use the
# simulated response in the next cell.

def call_bedrock(prompt, model_id='anthropic.claude-sonnet-4-20250514'):
    """Call Amazon Bedrock with the adjustment prompt."""
    # Model selection — update model_id to latest available Sonnet version as needed
    client = boto3.client('bedrock-runtime', region_name='us-east-1')
    
    response = client.invoke_model(
        modelId=model_id,
        contentType='application/json',
        accept='application/json',
        body=json.dumps({
            'anthropic_version': 'bedrock-2023-05-31',
            'max_tokens': 500,
            'temperature': 0.2,  # Low temperature for consistent analytical output
            'messages': [
                {'role': 'user', 'content': prompt}
            ]
        })
    )
    
    result = json.loads(response['body'].read())
    return json.loads(result['content'][0]['text'])

# Uncomment to run against Bedrock:
# adjustment = call_bedrock(prompt)
# print(json.dumps(adjustment, indent=2))

print("Bedrock call function defined. Uncomment above to execute with live credentials.")
print("Using simulated response below for demonstration.")

In [ ]:
# Simulated Bedrock response (representative of actual model output)
adjustment = {
    "adjustment_pct": 14,
    "confidence": 0.78,
    "reasoning": "Three corroborating signals point to a near-term demand surge: "
                 "viral social media exposure in the exact metro area (340% mention increase), "
                 "a sustained heat wave driving cold beverage consumption (+8-12% historical), "
                 "and competitor stockouts removing substitution options for 2 weeks. "
                 "No offsetting signals (no price increase, no negative press).",
    "primary_driver": "social_media + competitor_stockout combination",
    "risk_factors": "TikTok virality may not convert to purchase behavior in the "
                    "24-48 hour window needed to affect this forecast period. "
                    "Heat wave forecasts beyond 5 days carry uncertainty."
}

print("Foundation Model Recommendation:")
print(f"  Adjustment: +{adjustment['adjustment_pct']}%")
print(f"  Confidence: {adjustment['confidence']}")
print(f"  Primary driver: {adjustment['primary_driver']}")
print(f"\nReasoning: {adjustment['reasoning']}")
print(f"\nRisk factors: {adjustment['risk_factors']}")

## 5. Combine Forecasts

The final forecast is the statistical baseline adjusted by the foundation model's recommendation. The confidence score determines whether this executes autonomously or routes to a human planner.

In this case:
- Confidence (0.78) is above our autonomous threshold (0.70)
- Adjustment (+14%) is within the +/- 15% guardrail
- The system would execute this adjustment without human intervention

In [ ]:
# Apply adjustment to statistical forecast
adjusted_forecast = [f * (1 + adjustment['adjustment_pct'] / 100) for f in statistical_forecast]

# Decision logic
CONFIDENCE_THRESHOLD = 0.70
ADJUSTMENT_GUARDRAIL = 15  # percent

autonomous_execution = (
    adjustment['confidence'] >= CONFIDENCE_THRESHOLD and
    abs(adjustment['adjustment_pct']) <= ADJUSTMENT_GUARDRAIL
)

print("=" * 60)
print("FORECAST COMPARISON")
print("=" * 60)
print(f"\nSKU: SKU-4471-OATMILK-32OZ")
print(f"Location: DC-CHI-01")
print(f"Forecast horizon: 4 weeks")
print(f"\n{'Week':<10}{'Statistical':<15}{'Adjusted':<15}{'Delta':<10}")
print("-" * 50)
for i in range(forecast_weeks):
    delta = adjusted_forecast[i] - statistical_forecast[i]
    print(f"Week {i+1:<5}{statistical_forecast[i]:<15.0f}{adjusted_forecast[i]:<15.0f}+{delta:<10.0f}")

total_delta = sum(adjusted_forecast) - sum(statistical_forecast)
print(f"\nTotal additional units over 4 weeks: +{total_delta:.0f}")
print(f"\nExecution decision: {'AUTONOMOUS' if autonomous_execution else 'ESCALATE TO PLANNER'}")
print(f"  Confidence ({adjustment['confidence']}) {'≥' if adjustment['confidence'] >= CONFIDENCE_THRESHOLD else '<'} threshold ({CONFIDENCE_THRESHOLD})")
print(f"  Adjustment ({adjustment['adjustment_pct']}%) {'≤' if abs(adjustment['adjustment_pct']) <= ADJUSTMENT_GUARDRAIL else '>'} guardrail ({ADJUSTMENT_GUARDRAIL}%)")

In [ ]:
# Visualize: historical + both forecasts
fig, ax = plt.subplots(figsize=(12, 5))

# Historical
ax.plot(df['week_start'], df['units_sold'], marker='.', linewidth=1.5,
        color='#232F3E', markersize=4, label='Historical demand')

# Statistical forecast
ax.plot(forecast_dates, statistical_forecast, '--', linewidth=2,
        color='#666666', marker='s', markersize=6, label='Statistical forecast')

# Adjusted forecast
ax.plot(forecast_dates, adjusted_forecast, '-', linewidth=2.5,
        color='#FF9900', marker='^', markersize=8, label=f'FM-adjusted (+{adjustment["adjustment_pct"]}%)')

# Confidence band
upper = [f * 1.1 for f in adjusted_forecast]
lower = [f * 0.9 for f in adjusted_forecast]
ax.fill_between(forecast_dates, lower, upper, alpha=0.1, color='#FF9900')

ax.axvline(x=dates[-1], color='#999999', linestyle=':', alpha=0.7, label='Forecast start')
ax.set_xlabel('Week')
ax.set_ylabel('Units')
ax.set_title('Demand Forecast: Statistical vs. FM-Adjusted — SKU-4471 @ DC-CHI-01')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Key Takeaways

**What the foundation model adds:**
- Ability to interpret unstructured signals that statistical models cannot ingest
- Natural language reasoning traces that explain *why* an adjustment was made (critical for planner trust)
- Confidence scoring that enables automated guardrails

**What the foundation model does NOT replace:**
- The statistical baseline. FM-generated forecasts without a statistical anchor hallucinate numbers. The model adjusts, it does not predict.
- Domain-specific business rules (minimum order quantities, shelf-life constraints, contractual allocation minimums)
- Human judgment for high-stakes decisions above the confidence/guardrail thresholds

**Production considerations:**
- Prompt engineering matters enormously. The constraint to respond in structured JSON with bounded adjustment ranges prevents runaway recommendations.
- Temperature should be low (0.1-0.3) for analytical tasks. Higher temperatures produce inconsistent outputs that erode planner trust.
- Signal quality is the bottleneck, not model capability. Garbage signals produce garbage adjustments regardless of how capable the reasoning model is.
- Cost per decision at scale: ~$0.01-0.03 per SKU-location adjustment (2K input tokens + 500 output tokens on Claude Sonnet). For 5,000 SKUs across 12 DCs updated weekly, that is roughly $600-$1,800/month for the inference layer alone.